# 🗿 modelo3d — De foto a modelo 3D imprimible

Convertí una foto (o tres vistas del mismo objeto) en un archivo **STL listo para imprimir**, sin saber nada de programación.

## Qué necesitás
- Una cuenta de Google (gratis).
- Una foto del objeto: buena luz, fondo liso, un solo objeto centrado.

## Cuánto tarda
- **Primera vez:** 5–8 minutos de instalación automática (solo una vez por sesión).
- **Cada modelo:** entre 30 segundos y 2 minutos.

## Antes de empezar
1. Hacé clic en **Entorno de ejecución → Cambiar tipo de entorno de ejecución → T4 GPU → Guardar**.
2. Ejecutá la celda de instalación de abajo y esperá el mensaje ✅.
3. La aplicación va a aparecer al final de la página.

⚠️ **Importante:** cuando la sesión de Colab se cierre, los archivos se borran. Descargá tu STL apenas lo generes.


In [ ]:
# --- Instalación (ejecutar primero) ---
import sys

if not __import__("torch").cuda.is_available():
    raise RuntimeError(
        "GPU no activada. Andá a 'Entorno de ejecución → Cambiar tipo de entorno "
        "de ejecución → T4 GPU', guardá, y volvé a ejecutar esta celda."
    )

print("✅ GPU detectada:", __import__("torch").cuda.get_device_name(0))

REPO_URL = "https://github.com/Tencent-Hunyuan/Hunyuan3D-2.git"
REPO_DIR = "/content/Hunyuan3D-2"
MODEL_SINGLE = ("tencent/Hunyuan3D-2mini", "hunyuan3d-dit-v2-mini-turbo")
MODEL_MULTI = ("tencent/Hunyuan3D-2mv", "hunyuan3d-dit-v2-mv")
HF_REVISIONS = {
    "mini": "f90a0f7df7d5e6f71109cf333f6a95a0ae3194a6",
    "mv": "3a761b539b29fe4ff64714813aa9560fd66f5de0",
}

import subprocess  # noqa: E402

subprocess.run(
    ["bash", "-lc",
     f'test -d {REPO_DIR} || git clone --depth 1 {REPO_URL} {REPO_DIR}'],
    check=True,
)
subprocess.run(
    ["bash", "-lc",
     "pip install -q "
     "trimesh pymeshfix manifold3d gradio "
     "diffusers transformers accelerate "
     "omegaconf einops pygltflib xatlas"],
    check=True,
)
sys.path.insert(0, REPO_DIR)

from huggingface_hub import snapshot_download  # noqa: E402

print("⬇️ Descargando modelo de una foto (~1 GB)...")
snapshot_download(
    repo_id=MODEL_SINGLE[0],
    revision=HF_REVISIONS["mini"],
    allow_patterns=[f"{MODEL_SINGLE[1]}/*"],
)
print("⬇️ Descargando modelo multivista (~2 GB)...")
snapshot_download(
    repo_id=MODEL_MULTI[0],
    revision=HF_REVISIONS["mv"],
    allow_patterns=[f"{MODEL_MULTI[1]}/*"],
)

_ENGINES = None


def ensure_engines():
    """Carga diferida de los pipelines (solo forma, sin texturas)."""
    global _ENGINES
    if _ENGINES is None:
        from hy3dgen.shapegen import Hunyuan3DDiTFlowMatchingPipeline

        mini = Hunyuan3DDiTFlowMatchingPipeline.from_pretrained(
            MODEL_SINGLE[0], subfolder=MODEL_SINGLE[1]
        )
        mini.to("cuda")
        try:
            mv = Hunyuan3DDiTFlowMatchingPipeline.from_pretrained(
                MODEL_MULTI[0], subfolder=MODEL_MULTI[1]
            )
            mv.to("cuda")
        except Exception:
            print("⚠️ Motor multivista no disponible; se usará el de una foto.")
            mv = None
        _ENGINES = (mini, mv)
    return _ENGINES


print("✅ Instalación lista. Ejecutá la celda de abajo para abrir la app.")


In [ ]:
# --- Configuración general ---
SIZE_PRESETS_MM = {"10cm": 100, "15cm": 150}


def resolve_size_preset(preset: str, custom_mm: int | None) -> int:
    """Devuelve la altura objetivo en milímetros."""
    if preset in SIZE_PRESETS_MM:
        return SIZE_PRESETS_MM[preset]
    if preset == "custom":
        if not custom_mm or custom_mm <= 0:
            raise ValueError("Ingresá un alto en milímetros válido (mayor a 0).")
        return int(custom_mm)
    raise ValueError(f"Tamaño desconocido: {preset}")

In [ ]:
# --- Validación de fotos y mensajes ---
import numpy as np
from PIL import Image

MIN_SIDE_PX = 256

ERRORS_ES = {
    "too_small": "La foto es muy chica. Usá una imagen de al menos 256 píxeles por lado.",
    "unreadable": "No pudimos leer la imagen. Probá con otro archivo JPG o PNG.",
    "no_object": "No detectamos ningún objeto en la foto. Revisá que el objeto se vea completo y con buen contraste contra el fondo.",
    "bad_cutout": "El recorte del objeto quedó raro. Sacá la foto con el objeto centrado sobre un fondo liso, sin manos y sin que se corte con el borde.",
}

FALLBACK_ERROR_ES = (
    "Algo salió mal generando el modelo. Probá de nuevo; si sigue fallando, "
    "probá con otra foto."
)

REPAIR_ERROR_ES = "El modelo salió con agujeros que no pudimos reparar. Probá generar de nuevo con otra foto."


def _to_rgb(img: Image.Image) -> Image.Image:
    return img.convert("RGB") if img.mode != "RGB" else img


def validate_image(img: Image.Image) -> None:
    try:
        img = _to_rgb(img)
        w, h = img.size
    except Exception as exc:
        raise ValueError(ERRORS_ES["unreadable"]) from exc
    if w < MIN_SIDE_PX or h < MIN_SIDE_PX:
        raise ValueError(ERRORS_ES["too_small"])


def mask_fraction(mask: np.ndarray) -> float:
    return float(np.count_nonzero(mask)) / float(mask.size)


def check_mask_sane(fraction: float) -> None:
    if fraction < 0.01:
        raise ValueError(ERRORS_ES["no_object"])
    if fraction > 0.90:
        raise ValueError(ERRORS_ES["bad_cutout"])


def friendly_error(exc: Exception) -> str:
    if isinstance(exc, ValueError):
        msg = str(exc)
        if msg in ERRORS_ES.values():
            return msg
        if "agujeros" in msg:
            return REPAIR_ERROR_ES
    return FALLBACK_ERROR_ES


In [ ]:
# --- Núcleo geométrico: reparar, escalar, base, exportar ---
import numpy as np
import trimesh
import trimesh.boolean

BASE_HEIGHT_MM = 3.0
BASE_MARGIN = 0.95  # radio del pedestal relativo al alcance XY del modelo


def repair_mesh(mesh: trimesh.Trimesh) -> trimesh.Trimesh:
    mesh = mesh.copy()
    mesh.merge_vertices()
    mesh.update_faces(mesh.nondegenerate_faces())
    mesh.update_faces(mesh.unique_faces())
    mesh.remove_unreferenced_vertices()
    if not mesh.is_watertight:
        try:
            import pymeshfix

            verts = np.asarray(mesh.vertices, dtype=np.float64).copy()
            faces = np.asarray(mesh.faces, dtype=np.int32).copy()
            fixed = pymeshfix.clean_from_arrays(verts, faces)
            mesh = trimesh.Trimesh(fixed[0], fixed[1], process=False)
        except Exception as exc:
            raise ValueError(
                "El modelo salió con agujeros que no pudimos reparar. "
                "Probá generar de nuevo con otra foto."
            ) from exc
    if mesh.volume < 0:
        mesh.invert()
    return mesh


def normalize_mesh(
    mesh: trimesh.Trimesh, target_height_mm: float
) -> trimesh.Trimesh:
    m = mesh.copy()
    height = float(m.extents[2])
    if height <= 0:
        raise ValueError("La geometría generada es plana e inválida.")
    m.apply_scale(target_height_mm / height)
    m.apply_translation(-m.bounds[0])          # apoyar en Z=0
    center_xy = m.bounds.mean(axis=0)[:2]
    m.apply_translation([-center_xy[0], -center_xy[1], 0])
    return m


def _pedestal(radius_mm: float, height_mm: float) -> trimesh.Trimesh:
    cyl = trimesh.creation.cylinder(
        radius=radius_mm, height=height_mm, sections=64
    )
    cyl.apply_translation([0, 0, height_mm / 2.0])
    return cyl


def add_flat_base(
    mesh: trimesh.Trimesh, height_mm: float = BASE_HEIGHT_MM
) -> trimesh.Trimesh:
    xy_span = float(max(mesh.extents[0], mesh.extents[1]))
    pedestal = _pedestal(xy_span * BASE_MARGIN * 0.5, height_mm)
    merged = trimesh.boolean.union([mesh, pedestal], engine="manifold")
    return merged


def export_stl(mesh: trimesh.Trimesh, path: str) -> str:
    mesh.export(path, file_type="stl")
    return path


def verify_stl(
    path: str, target_height_mm: float, tol_mm: float = 0.5
) -> trimesh.Trimesh:
    reloaded = trimesh.load(path, force="mesh")
    if not reloaded.is_watertight:
        raise ValueError("El STL exportado tiene agujeros.")
    if reloaded.volume <= 0:
        raise ValueError("El STL exportado está vacío.")
    height = float(reloaded.extents[2])
    if abs(height - target_height_mm) > tol_mm:
        raise ValueError(
            f"El STL mide {height:.1f} mm de alto en vez de {target_height_mm:.1f} mm."
        )
    return reloaded


In [ ]:
# --- Selección de motor y reintento por memoria ---
GEN_PARAMS = {"num_inference_steps": 50, "octree_resolution": 256}
GEN_PARAMS_FAST = {"num_inference_steps": 30, "octree_resolution": 192}


def choose_engine(n_views: int) -> str:
    return "mv" if n_views >= 2 else "single"


def select_mv_strategy(mv_available: bool, n_views: int) -> tuple[str, str]:
    wanted = choose_engine(n_views)
    if wanted == "mv" and not mv_available:
        return "single", (
            "El modo varias fotos no está disponible en esta sesión; "
            "usamos el motor de una sola foto."
        )
    return wanted, ""


def generate_with_retry(run_fn, params: dict, oom_exc: type):
    try:
        import torch
        oom = oom_exc if oom_exc is not Exception else getattr(
            torch.cuda, "OutOfMemoryError", RuntimeError
        )
    except ImportError:
        oom = oom_exc

    try:
        return run_fn(params)
    except oom:
        print("⚠️ Sin memoria suficiente; reintentando en calidad reducida...")
        import gc

        gc.collect()
        try:
            import torch
            torch.cuda.empty_cache()
        except (ImportError, AttributeError):
            pass
        return run_fn({**params, **GEN_PARAMS_FAST})


In [ ]:
# --- Aplicación web ---
import os
import tempfile
import gradio as gr

PHOTO_TIPS_ES = (
    "- Buena luz, sin flash directo\n"
    "- Fondo liso y de color parejo\n"
    "- Un solo objeto, centrado y completo\n"
    "- Sin manos sosteniendo el objeto\n"
    "- En modo varias fotos: misma distancia y altura en las tres tomas"
)

_REMOVER = None


def _segment(img):
    """Quita el fondo y devuelve (imagen RGB, máscara bool)."""
    global _REMOVER
    if _REMOVER is None:
        from hy3dgen.shapegen.rembg import BackgroundRemover
        _REMOVER = BackgroundRemover()
    remover = _REMOVER
    out = remover(img)
    import numpy as np

    rgba = np.array(out.convert("RGBA"))
    mask = rgba[..., 3] > 127
    return out.convert("RGB"), mask


def _call_engine(engine, images, progress):
    mini, mv = ensure_engines()

    def run(params):
        progress((0.4, "Generando geometría…"))
        if engine == "mv":
            mesh = mv(image=images, **params)[0]
        else:
            mesh = mini(image=images[0], **params)[0]
        return mesh.to_data() if hasattr(mesh, "to_data") else mesh

    import torch

    raw = generate_with_retry(run, GEN_PARAMS, oom_exc=torch.cuda.OutOfMemoryError)
    import trimesh

    return raw if isinstance(raw, trimesh.Trimesh) else trimesh.Trimesh(
        raw.vertices.detach().cpu().numpy(),
        raw.faces.detach().cpu().numpy(),
        process=False,
    )


def run_pipeline(single, front, left, back, modo, preset, custom_mm,
                 want_base, progress=gr.Progress()):
    imgs = []
    if modo == "Varias fotos":
        imgs = [im for im in (front, left, back) if im is not None]
        if not imgs:
            imgs = [single]
    else:
        imgs = [single]

    progress((0.05, "Revisando la foto…"))
    validate_image(imgs[0])
    seg = [_segment(im) for im in imgs]
    for _, mask in seg:
        check_mask_sane(mask_fraction(mask))
    imgs = [im for im, _ in seg]

    mini, mv = ensure_engines()
    engine, notice = select_mv_strategy(mv is not None, len(imgs))

    mesh = _call_engine(engine, imgs, progress)
    progress((0.7, "Reparando la malla…"))
    mesh = repair_mesh(mesh)

    target_mm = resolve_size_preset(preset, custom_mm)
    mesh = normalize_mesh(mesh, target_mm)
    if want_base:
        mesh = add_flat_base(mesh)

    progress((0.9, "Exportando STL…"))
    workdir = tempfile.mkdtemp(prefix="modelo3d_")
    stl_path = export_stl(mesh, f"{workdir}/modelo.stl")
    verify_stl(stl_path, target_mm)
    glb_path = f"{workdir}/modelo.glb"
    mesh.export(glb_path, file_type="glb")

    status = "✅ Modelo listo para imprimir."
    if notice:
        status += f" ({notice})"
    return glb_path, stl_path, status


def build_app():
    with gr.Blocks(title="modelo3d") as demo:
        gr.Markdown("## 🗿 De foto a modelo 3D imprimible")
        with gr.Accordion("Consejos para la foto", open=False):
            gr.Markdown(PHOTO_TIPS_ES)
        modo = gr.Radio(["Una foto", "Varias fotos"],
                        value="Una foto", label="Modo")
        single = gr.Image(type="pil", label="Foto del objeto")
        with gr.Row(visible=False) as fila_multi:
            front = gr.Image(type="pil", label="Frente")
            left = gr.Image(type="pil", label="Perfil izquierdo")
            back = gr.Image(type="pil", label="Espalda")
        modo.change(lambda m: gr.update(visible=m == "Varias fotos"),
                    modo, fila_multi)
        tamano = gr.Dropdown(["10cm", "15cm", "custom"],
                             value="10cm", label="Tamaño impreso")
        custom_mm = gr.Number(label="Milímetros (si elegís custom)",
                              precision=0)
        base_chk = gr.Checkbox(value=True,
                               label="Agregar base plana (recomendado)")
        btn = gr.Button("Generar", variant="primary")
        preview = gr.Model3D(label="Vista previa")
        archivo = gr.File(label="Descargar STL")
        estado = gr.Markdown()

        def wrapped(*args):
            try:
                return run_pipeline(*args)
            except Exception as exc:  # nunca mostrar traceback crudo
                import traceback
                print("DEBUG ERROR:", type(exc).__name__, str(exc))
                traceback.print_exc()
                raise gr.Error(friendly_error(exc) + "\n\n[Debug] " + type(exc).__name__ + ": " + str(exc))

        btn.click(wrapped,
                  [single, front, left, back, modo, tamano, custom_mm,
                   base_chk],
                  [preview, archivo, estado])
    return demo


if "google.colab" in __import__("sys").modules or os.environ.get("MODELO3D_IN_COLAB") == "1":
    demo = build_app()
    demo.queue().launch(share=True, debug=False)


In [ ]:
# --- Modo prueba (recomendado antes del primer uso) ---
import os
from PIL import Image, ImageDraw


def make_sample_image() -> Image.Image:
    """Imagen sintética determinista: figura oscura sobre fondo claro."""
    img = Image.new("RGB", (768, 768), (235, 235, 230))
    d = ImageDraw.Draw(img)
    d.ellipse([284, 84, 484, 284], fill=(70, 55, 45))     # cabeza
    d.rounded_rectangle([264, 264, 504, 684], radius=80,
                        fill=(90, 75, 60))                 # cuerpo
    return img


def run_self_test() -> bool:
    print("🧪 SELF TEST — probando el pipeline completo…")
    img = make_sample_image()
    glb, stl, status = run_pipeline(
        img, None, None, None, "Una foto", "10cm", None, True
    )
    ok = bool(stl) and "✅" in status
    if ok:
        verify_stl(stl, 100)  # double-check STL integrity
    print("SELF TEST:", "PASS ✅" if ok else "FAIL ❌", "—", status)
    return ok


if os.environ.get("MODELO3D_IN_COLAB") == "1":
    run_self_test()
